# 3 · Searching a motif in hyperspace

In notebook 1, Cypher found the precedent for a sales order SO-1042 with an exact motif query: lamps, East
London depot, Ireland. In notebook 2, we saw the three HDC operators. In this notebook we
answer the same question a second way, with hypervectors.

Our design rule is as follows: **rather than encoding the items individually, we encode 1-hop paths in the Lance relationship tables.** Every relationship in those tables gets its own hypervector, stored as a column next to `src` and `dst`. When we have a query about a
pattern or motif, it doesn't need to be precomputed in storage. It's **composed on demand** from those edge hypervectors, for whatever
pattern the analyst asks about.

Why is this quality significant? Because the motif is an analyst's choice, and it's not fixed. Today, we might care about "lamps shipped from East
London to Ireland". Tomorrow, we might care about "this exact product from this depot", or something
nobody has thought of yet. If we encoded multi-hop path hypervectors and store them upfront, we'd bake today's
question into the data, without accounting for tomorrow's concerns.

HDC's operators are **composable**, which is one of its greatest properties.

## Three more terms

| Term | What it means |
|---|---|
| **edge hypervector** | One edge, encoded as its relationship type bound to the node it points to: $\mathbf{r} \otimes \rho(\mathbf{target})$. It's stored as a `vector` column on the edge's row. |
| **motif hypervector** | One order's pattern for one question: the paths the analyst picked, each composed from edge hypervectors, bundled together. It's built at query time and thrown away after. |
| **associative search** | Looking things up by *what they resemble*, not by an exact key. You give a partial or approximate pattern, and get back the candidates most similar to it. In HDC this is a nearest-neighbour search by cosine similarity, also called an *associative memory*. |

## Setup

`sales_orders.py` holds the graph from notebook 1 (`write_graph`, `edge_rows`, `graph_config`)
and the codebook from notebook 2 (`make_codebook`). The seed is fixed, so these are the same
hypervectors as in notebook 2. We'll open the graph engine once the edges have their
hypervectors.

In [1]:
from sales_orders import NODES, EDGES, write_graph, edge_rows, graph_config, make_codebook

import lancedb
import polars as pl
import torch
import torchhd
from lance_csr import CypherEngine

pl.Config.set_tbl_rows(20)  # print every row of our small tables
pl.Config.set_fmt_str_lengths(200)  # and the HAD_ISSUE notes in full

db = lancedb.connect("sales.lancedb")
write_graph(db)

codebook = make_codebook()
D = len(codebook["ireland"])

## What are we looking for?

**Which past sales orders, like SO-1042, contained lamps shipped from East London to Ireland?**
Each fact comes from a **path**, a sequence of graph edges. Each edge is one **hop**.

| Fact to compare | Follow these edges from the order | SO-1042's value |
|---|---|---|
| Product category | Order → product → category (2 hops) | lamps |
| Shipping depot | Order → depot (1 hop) | east-london |
| Destination country | Order → region → country (2 hops) | ireland |

A different product, customer or destination city can still match these three facts.

## Prepare the graph for search

### Encode a path

We encode a path such as "ships to a region in Ireland" using its relationship types and
its final node. This lets us distinguish paths that reach the same node through different
relationships.

As in notebook 2, we bind a key to a value. Here, the key represents the sequence of
relationship types, and the value is the final node. Permutation ρ marks each step's
position so that changing the relationship order changes the encoding:

$$\text{path}(r_1, \dots, r_k, n) = \mathbf{r_1} \otimes \rho(\mathbf{r_2}) \otimes \dots \otimes \rho^{k-1}(\mathbf{r_k}) \otimes \rho^k(\mathbf{n})$$

For example, $\text{path}(\text{SHIPS\_TO}, \text{IN\_COUNTRY}, \text{ireland}) = \mathbf{SHIPS\_TO} \otimes \rho(\mathbf{IN\_COUNTRY}) \otimes \rho^2(\mathbf{ireland})$.

In [2]:
def path(*steps):
    hv = codebook[steps[0]]
    for k, step in enumerate(steps[1:], start=1):
        hv = torchhd.bind(hv, torchhd.permute(codebook[step], shifts=k))
    return hv

to_ireland = path("SHIPS_TO", "IN_COUNTRY", "ireland")
to_ireland[:10].tolist()

[1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]

### Roles stay separate

Here's why the relationship types belong in the key. Suppose the retailer opens a depot in
Dublin. Then an order could reach the `ireland` node in two **roles**: shipped *to* Ireland,
or shipped *from* Ireland. Both paths end at the same node, but their hypervectors are
unrelated, so a search for one won't match the other.

In [3]:
from_ireland = path("SHIPS_FROM", "IN_COUNTRY", "ireland")
print(f"cos(shipped to Ireland, shipped from Ireland) = {torchhd.cosine_similarity(to_ireland, from_ireland).item():+.3f}")

cos(shipped to Ireland, shipped from Ireland) = +0.003


### Store the edge hypervectors

We store a hypervector on each edge. When we search, the graph retrieves the relevant
edges, and we compose their vectors into the paths we want to compare. This lets us
change the question without rebuilding the stored vectors.


In [4]:
name = {n["id"]: n["name"] for rows in NODES.values() for n in rows}

for rel in EDGES:
    rows = [row | {"vector": path(rel, name[row["dst"]]).tolist()} for row in edge_rows(rel)]
    db.create_table(rel, rows, mode="overwrite")

db.open_table("SHIPS_FROM").search().limit(3).to_polars()

src,dst,vector
i64,i64,"array[f32, 10000]"
901,401,"[1.0, 1.0, … 1.0]"
904,403,"[-1.0, -1.0, … -1.0]"
908,402,"[1.0, 1.0, … 1.0]"


Now we open the graph engine over the tables. The `vector` column is just one more edge
property, like `note`, so the CSR adjacency is the same as in notebook 1.

In [5]:
tables = {}
for table_name in [*NODES, *EDGES]:
    tables[table_name] = db.open_table(table_name).to_lance()

engine = CypherEngine(graph_config(), tables, use_csr=True)


def cypher(q):
    return engine.execute(q)

## Compose a motif for every past order

`path_vectors()` traverses a chosen path for past orders with issues and composes its
hypervector from the stored edges. `compose_motifs()` bundles the selected paths for each
order. These candidate motifs are built in memory for the current question.

In [7]:
def path_vectors(*rels):
    """Each past order's path along rels, composed from the vector stored on the path's last edge."""
    hops = "".join(f"-[:{r}]->()" for r in rels[:-1])
    q = f"MATCH (o:Order){hops}-[e:{rels[-1]}]->(), (o)-[:HAD_ISSUE]->() RETURN o.name AS order_id, e.vector AS vector"
    vectors = {}
    for row in cypher(q).to_pylist():
        hv = torch.tensor(row["vector"])
        for r in reversed(rels[:-1]):
            hv = torchhd.bind(codebook[r], torchhd.permute(hv, shifts=1))
        vectors[row["order_id"]] = hv
    return vectors


def compose_motifs(*paths):
    """One motif hypervector per past order: the bundle of its composed paths."""
    per_path = [path_vectors(*rels) for rels in paths]
    orders = sorted(set().union(*per_path))
    return {o: torchhd.multiset(torch.stack([vectors[o] for vectors in per_path if o in vectors])) for o in orders}

## Search with what we know

We return to our original question: **Which past sales orders, like SO-1042, contained lamps
shipped from East London to Ireland?**

We start with Ireland, then add East London and lamps. Each clue specifies a relationship sequence and its destination node.
`search()` bundles those clues, composes the same paths for past orders, and ranks the
candidates by cosine similarity. The helper traverses paths using Cypher; we don't write a
new matching or scoring query for each combination of clues.

![A query hypervector compared with three candidate hypervectors, with one match highlighted](../img/hdc-retrieval.png)

In [8]:
def search(question):
    query = torchhd.multiset(torch.stack([path(*steps) for steps in question]))
    candidates = compose_motifs(*(steps[:-1] for steps in question))
    scores = [
        {"order_id": order, "score": round(torchhd.cosine_similarity(query, motif).item(), 3)}
        for order, motif in candidates.items()
    ]
    return pl.DataFrame(scores).sort(["score", "order_id"], descending=[True, False])

## Start with the destination

We only know that SO-1042 ships to **Ireland**. Which past orders followed that path?
The table ranks all past cases, including those that don't match.

In [9]:
question = [("SHIPS_TO", "IN_COUNTRY", "ireland")]
by_country = search(question)
by_country

order_id,score
str,f64
"""SO-0901""",1.0
"""SO-0904""",1.0
"""SO-0908""",1.0
"""SO-0917""",1.0
"""SO-0920""",1.0
"""SO-0912""",0.008
"""SO-0915""",0.008
"""SO-0903""",0.004
"""SO-0906""",0.004


## Add the depot

Now we learn it ships from **East London**. Append that clue and compose the query again
from the same stored edges.

In [10]:
question.append(("SHIPS_FROM", "east-london"))
by_depot = search(question)
by_depot

order_id,score
str,f64
"""SO-0901""",1.0
"""SO-0917""",1.0
"""SO-0920""",1.0
"""SO-0908""",0.509
"""SO-0904""",0.5
"""SO-0903""",0.019
"""SO-0914""",0.019
"""SO-0906""",0.013
"""SO-0907""",0.013


## Add the product category

The order contains **lamps**. Add the path through the product to its category, without
specifying the exact product.

In [11]:
question.append(("CONTAINS", "IN_CATEGORY", "lamps"))
hits = search(question)
hits

order_id,score
str,f64
"""SO-0917""",1.0
"""SO-0901""",0.669
"""SO-0920""",0.667
"""SO-0903""",0.342
"""SO-0907""",0.342
"""SO-0904""",0.338
"""SO-0908""",0.338
"""SO-0912""",0.335
"""SO-0906""",0.025


In [12]:
assert hits["order_id"][0] == "SO-0917" and hits["score"][0] > 0.95
assert (hits["score"][1:] < 0.7).all()

SO-0917 comes first after all three clues. Scores are approximately the fraction of shared
paths: 1 for three, 0.67 for two, 0.33 for one, and near zero for none. These are similarity
scores, not probabilities.

## The same search in Cypher

The exact Cypher motif query returns HDC's top tier. The following `CASE` query also ranks
partial matches by counting the clues they share.

In [13]:
exact = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(:Category {name: 'lamps'}),
      (o)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[:HAD_ISSUE]->(:Issue)
RETURN o.name AS order_id
'''
exact_ids = set(cypher(exact).column("order_id").to_pylist())
top_tier = set(hits.filter(pl.col("score") > 0.9)["order_id"])
print("Cypher:", exact_ids, "  HDC top tier:", top_tier)
assert exact_ids == top_tier

Cypher: {'SO-0917'}   HDC top tier: {'SO-0917'}


To rank near misses, this Cypher query spells out one `CASE` per clue.

In [14]:
graded = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(c:Category),
      (o)-[:SHIPS_FROM]->(d:Depot),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(k:Country),
      (o)-[:HAD_ISSUE]->(:Issue)
RETURN o.name AS order_id,
       (CASE WHEN c.name = 'lamps' THEN 1 ELSE 0 END)
     + (CASE WHEN d.name = 'east-london' THEN 1 ELSE 0 END)
     + (CASE WHEN k.name = 'ireland' THEN 1 ELSE 0 END) AS matches
ORDER BY matches DESC, order_id
'''
counts = pl.from_arrow(cypher(graded))
counts

order_id,matches
str,i64
"""SO-0917""",3
"""SO-0901""",2
"""SO-0920""",2
"""SO-0903""",1
"""SO-0904""",1
"""SO-0907""",1
"""SO-0908""",1
"""SO-0912""",1
"""SO-0906""",0


Compare the matching fraction with HDC's similarity score. Small differences come from
the random hypervectors.

In [15]:
side_by_side = counts.join(hits, on="order_id", maintain_order="left").with_columns(
    (pl.col("matches") / 3).round(3).alias("matches / 3")
)
side_by_side

order_id,matches,score,matches / 3
str,i64,f64,f64
"""SO-0917""",3,1.0,1.0
"""SO-0901""",2,0.669,0.667
"""SO-0920""",2,0.667,0.667
"""SO-0903""",1,0.342,0.333
"""SO-0904""",1,0.338,0.333
"""SO-0907""",1,0.342,0.333
"""SO-0908""",1,0.338,0.333
"""SO-0912""",1,0.335,0.333
"""SO-0906""",0,0.025,0.0


In [16]:
assert ((side_by_side["score"] - side_by_side["matches / 3"]).abs() < 0.05).all()

## A new question requires a new motif

**Which past sales orders contained the cordless table lamp shipped from East London,
regardless of destination?**

We now ask for the exact product, rather than any lamp, and drop the Ireland requirement.
We change the path list and reuse the saem stored edges in LanceDB. Note that we did not need to re-encode any of the underlying data.


In [17]:
new_question = [
    ("CONTAINS", "cordless-table-lamp"),
    ("SHIPS_FROM", "east-london"),
]
by_product = search(new_question)
by_product

order_id,score
str,f64
"""SO-0917""",1.0
"""SO-0901""",0.495
"""SO-0920""",0.495
"""SO-0903""",0.016
"""SO-0908""",0.014
"""SO-0914""",0.014
"""SO-0915""",0.007
"""SO-0906""",0.003
"""SO-0904""",0.001


In [19]:
assert by_product["order_id"][0] == "SO-0917" and by_product["score"][0] > 0.95

### What the results show

- **SO-0917 scores 1.0** because it matches both the product and depot.
- **SO-0901 and SO-0920 score about 0.5** because they match the depot but contain different products.
- **The other orders score near zero** because they match neither clue.

As our needs evolved, we changed what we retrieved by changing how we encode the query's paths. The stored data stayed the same, and the search still ranked both full and partial matches that surfaced useful results.


## From a candidate to the evidence

Similarity tells our colleague *which* past cases to look at, not what happened in them. For
that, we go back to the graph and read the `HAD_ISSUE` notes of the two top tiers.

In [19]:
shortlist = hits.filter(pl.col("score") > 0.5)["order_id"].to_list()
# An f-string is fine for names we produced ourselves; never build Cypher from user input this way.
q = f'''
MATCH (o:Order)-[h:HAD_ISSUE]->(i:Issue)
WHERE o.name IN {shortlist}
RETURN o.name, i.name, h.note
'''
pl.from_arrow(cypher(q))

o.name,i.name,h.note
str,str,str
"""SO-0901""","""customs_delay""","""Commercial invoice missing; Irish customs held the parcel until the broker resent it."""
"""SO-0917""","""dispatch_hold""","""Cordless lamps (lithium batteries) held at East London depot for dangerous-goods paperwork; released after calling the depot."""
"""SO-0920""","""customs_delay""","""Customs declaration lodged late by the broker."""


The closest case describes a **dispatch hold** for battery paperwork at East London.
The near misses describe customs delays. These are leads to investigate, not proof of
what happened to SO-1042.

## Wrap-up

We stored edge vectors, then composed paths, candidate motifs and queries at runtime.
Adding clues refined the ranking without a new hand-written scoring query. Changing the
path list let us ask about the exact product instead of its category.

In notebook 4, we use these past cases to build a memory of issue types.